In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_03_telemetry (parameterized)
# Schedule: Every 2 min (test) / 30 min (prod)

import uuid, random, datetime as dt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
ROOT = LANDING
SEED = f"{ROOT}/_seed"

# SAFETY: seed must exist
try:
    machines = [r.asDict() for r in spark.read.json(f"{SEED}/machines").collect()]
    if not machines: raise Exception()
    print(f"✅ Seed loaded: {len(machines)} machines")
except:
    dbutils.notebook.exit("❌ SEED MISSING — run sim_00_seed first")

machine_ids = [m["machine_id"] for m in machines]

run_ts = dt.datetime.utcnow()
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"

# ---- sensor_event (300-500 per run) ----
sensor = []
for _ in range(random.randint(300, 500)):
    mid = random.choice(machine_ids)
    anomaly = random.random() < 0.05
    sensor.append({
        "event_id": f"EVT-{uuid.uuid4().hex[:10]}",
        "machine_id": mid,
        "sensor_ts": run_ts.isoformat()+"Z",
        "engine_temp": round(random.uniform(121,140) if anomaly else random.uniform(80,100), 2),
        "hydraulic_press": round(random.uniform(50,78) if anomaly else random.uniform(180,220), 2),
        "fuel_level": round(random.uniform(0.1,1.0), 3),
        "rpm": random.randint(1400, 2600),
        "fault_code": random.choice([None]*19 + [f"F{random.randint(100,999)}"]),
    })
spark.createDataFrame(sensor).write.mode("append").json(
    f"{ROOT}/telemetry/sensor_event/{run_id}.json")
print(f"  ✓ sensor_event: {len(sensor)}")

# ---- fault_event (5-20 per run) ----
faults = [{
    "fault_event_id": f"FLT-{uuid.uuid4().hex[:10]}",
    "machine_id": random.choice(machine_ids),
    "fault_code": f"F{random.randint(100,999)}",
    "severity": random.choice(["low","medium","high","critical"]),
    "fault_ts": run_ts.isoformat()+"Z",
} for _ in range(random.randint(5, 20))]
spark.createDataFrame(faults).write.mode("append").json(
    f"{ROOT}/telemetry/fault_event/{run_id}.json")
print(f"  ✓ fault_event: {len(faults)}")

# ---- engine_hours (20 machines per run) ----
eh = [{
    "record_id": f"EH-{uuid.uuid4().hex[:10]}",
    "machine_id": m,
    "cumulative_hours": round(random.uniform(1000,20000),2),
    "record_ts": run_ts.isoformat()+"Z",
} for m in random.sample(machine_ids, k=min(20, len(machine_ids)))]
spark.createDataFrame(eh).write.mode("append").json(
    f"{ROOT}/telemetry/engine_hours/{run_id}.json")
print(f"  ✓ engine_hours: {len(eh)}")

print(f"\n✅ Telemetry run complete: {run_id}")